# TPPP report — `tppp_report_v5.ipynb`

Builds **one self-contained HTML file** for the team behind the TPPP use case, from the outputs of
`tppp_prospects_v5.ipynb`. No model is rerun; this notebook only reads, selects and presents.

| Input | From v5 |
|---|---|
| `pkg_tppp_cust_scores` | look-alikes: scores, triage, evidence |
| `pkg_tppp_cpty_scores` | prospects: scores, relationship to PNC, evidence, value definitions and tiers |
| `OUT_DIR/tppp_prospects_ranked.csv` | warm introductions for tiers A–B |
| `OUT_DIR/tppp_registered_business_elsewhere_monthly.csv` | registered TPPPs month by month |
| `OUT_DIR/cust_recall.csv`, `cpty_recall.csv`, `evidence_checks.csv` | how well the method and the checks work |

**The report** opens with headline numbers and three starting points (one per audience), then: registered TPPPs'
business at other banks, ranked prospects with warm introductions, look-alikes and the KYC check list, card settlement
at other banks, how reliable the results are, and the method in plain words. Tables can be searched and sorted; it
works offline (no external scripts or fonts) and follows the reader's light or dark setting.

**It contains client names and power ids**: share it only inside the TPPP / TM working group until the
permissible-use review is done.

In [ ]:
# §0  Setup and config
import os, re, json, math, html, datetime as dt, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                            ("uint0", np.uintp), ("bool", np.bool_), ("object", object)]:
        if not hasattr(np, _alias):
            setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
from IPython.display import display, HTML
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException

spark = (SparkSession.builder.appName("pkg_tppp_report")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .enableHiveSupport().getOrCreate())

DB           = "bdahd01p_dlcdi1_cdi_tm"
T_CUST       = f"{DB}.pkg_tppp_cust_scores"
T_CPTY       = f"{DB}.pkg_tppp_cpty_scores"
T_INTRO      = f"{DB}.pkg_tppp_cpty_intros"
OUT_DIR      = "../metrics/tppp/v5"          # where tppp_prospects_v5.ipynb wrote its CSVs
REPORT_FILE  = f"{OUT_DIR}/tppp_report_v5.html"
STABLE_END   = "2026-06"                     # amounts quoted over months ≤ this (July–August 2026 source jump)
N_PROSPECTS  = 150                           # prospects table (tiers A–B first)
N_LOOKALIKES = 100
N_DECLARED   = 40
N_BRANDS     = 30
N_BANKS      = 15
SCATTER_MAX  = 4000                          # points in the prospect map (all A and B are always kept)
TITLE        = "TPPP opportunities in the payment graph"
AUDIENCE     = "TPPP / Treasury Management working group"

def table_ok(t):
    try:
        spark.table(t).columns
        return True
    except AnalysisException:
        return False

def show(title, *blocks):
    css = ("<style>.rt{border-collapse:collapse;font-size:12px}.rt th,.rt td{padding:2px 8px;border-bottom:1px solid #ddd;"
           "text-align:left}.rt th{background:#e3e2de;color:#0b0b0b}</style>")
    parts = [f"<div class='tex2jax_ignore mathjax_ignore'><b style='font-size:14px'>{html.escape(title)}</b>"]
    for name, df in blocks:
        if df is None:
            continue
        if isinstance(df, str):
            parts.append(f"<div style='color:#6f6e69;margin:4px 0'>{html.escape(df)}</div>")
        else:
            parts.append(f"<div style='margin:10px 0 2px'><b>{html.escape(name)}</b></div>"
                         + df.to_html(index=False, border=0, classes="rt"))
    display(HTML(css + "".join(parts) + "</div>"))
print(f"Spark {spark.version} | report → {REPORT_FILE}")

In [ ]:
# §1  Load the v5 outputs
need = {T_CUST: ["customer_name", "customer_pwr_id", "naics_cd", "naics_desc", "is_tppp", "tppp_family", "excluded",
                 "sector_unlikely", "declared", "triage", "score", "screened_rank", "reasons", "evidence", "evidence_n",
                 "data_confirmed", "nearest_tppp", "amt_in", "amt_out", "ncpty_in", "ncpty_out", "window", "version"],
        T_CPTY: ["ent_id", "ent_name", "fi_dom", "known_proc", "tppp_ext", "cpty_excl", "card_settlement", "relationship",
                 "pnc_match_names", "evidence", "evidence_n", "data_confirmed", "score", "amt_in", "amt_out", "ncust_in",
                 "ncust_out", "ncust_tm_in", "ncust_tm_out", "amt_stable", "growth", "p_wallet", "p_reach", "p_steady",
                 "p_growth", "p_fit", "consensus", "tier", "value_rank", "value_reason"]}
rows = []
for t, cols in need.items():
    assert table_ok(t), f"{t} not found: run tppp_prospects_v5.ipynb first"
    miss = [c for c in cols if c not in spark.table(t).columns]
    assert not miss, f"{t} lacks {miss}: it was written by an older version; rerun tppp_prospects_v5.ipynb"
CS = spark.table(T_CUST).toPandas()
CP = spark.table(T_CPTY).toPandas()
INTRO = spark.table(T_INTRO).toPandas() if table_ok(T_INTRO) else pd.DataFrame()
rows += [{"input": T_CUST, "rows": len(CS)}, {"input": T_CPTY, "rows": len(CP)}, {"input": T_INTRO, "rows": len(INTRO)}]

def read_csv(name):
    p = f"{OUT_DIR}/{name}"
    df = pd.read_csv(p) if os.path.exists(p) else None
    rows.append({"input": p, "rows": len(df) if df is not None else "missing (section left out)"})
    return df
RANKED  = read_csv("tppp_prospects_ranked.csv")
MONTHLY = read_csv("tppp_registered_business_elsewhere_monthly.csv")
REC_C   = read_csv("cust_recall.csv")
REC_P   = read_csv("cpty_recall.csv")
CHECKS  = read_csv("evidence_checks.csv")
for D in (CS, CP):
    for c in [c for c in D.columns if c in ("is_tppp", "tppp_family", "excluded", "sector_unlikely", "declared",
                                             "data_confirmed", "known_proc", "tppp_ext", "cpty_excl", "card_settlement")]:
        D[c] = D[c].fillna(False).astype(bool)
WINDOW_TXT = str(CS.window.dropna().iloc[0]) if CS.window.notna().any() else ""
VERSION = str(CS.version.dropna().iloc[0]) if CS.version.notna().any() else ""
show(f"§1 Inputs (run {VERSION}, window {WINDOW_TXT})", ("Loaded", pd.DataFrame(rows)))

In [ ]:
# §2  Select what the report shows
# ---- registered TPPPs: business at other banks (stable months only) ----
REG = None
if MONTHLY is not None and len(MONTHLY):
    mcols = [c for c in MONTHLY.columns if re.match(r"^\d{4}-\d{2} ", c)]
    st_cols = [c for c in mcols if c[:7] <= STABLE_END]
    lt_cols = [c for c in mcols if c[:7] > STABLE_END]
    m = MONTHLY.copy()
    m["stable"] = m[st_cols].mean(axis=1)
    m["late"] = m[lt_cols].mean(axis=1) if lt_cols else np.nan
    w = m.pivot_table(index="TPPP", columns="PNC clients pay into", values=["stable", "late"], aggfunc="sum")
    REG = pd.DataFrame({"tppp": w.index,
                        "pnc": w[("stable", "PNC accounts")].values * 1e6,
                        "out": w[("stable", "elsewhere")].values * 1e6,
                        "pnc_late": w[("late", "PNC accounts")].values * 1e6 if lt_cols else np.nan,
                        "out_late": w[("late", "elsewhere")].values * 1e6 if lt_cols else np.nan})
    REG["share"] = REG.out / (REG.out + REG.pnc)
    REG["share_late"] = REG.out_late / (REG.out_late + REG.pnc_late)
    trend = {}
    for t_, g_ in m[m["PNC clients pay into"] == "elsewhere"].groupby("TPPP"):
        p_ = m[(m.TPPP == t_) & (m["PNC clients pay into"] == "PNC accounts")]
        if len(p_) and len(st_cols) >= 2:
            a0, b0 = g_[st_cols[0]].iloc[0], p_[st_cols[0]].iloc[0]
            a1, b1 = g_[st_cols[-1]].iloc[0], p_[st_cols[-1]].iloc[0]
            trend[t_] = (a0 / (a0 + b0) if a0 + b0 else np.nan, a1 / (a1 + b1) if a1 + b1 else np.nan)
    REG["share_first"] = REG.tppp.map(lambda t: trend.get(t, (np.nan, np.nan))[0])
    REG["share_last"] = REG.tppp.map(lambda t: trend.get(t, (np.nan, np.nan))[1])
    REG = REG.sort_values("out", ascending=False).reset_index(drop=True)
    MONTHS_ST = [c[:7] for c in st_cols]

# ---- prospects ----
G = CP[CP.tier.notna()].sort_values("value_rank").copy()
G["tm_clients"] = G.ncust_tm_in.fillna(0) + G.ncust_tm_out.fillna(0)
if RANKED is not None and "warm intros" in RANKED.columns:
    G["intros"] = G.ent_id.map(dict(zip(RANKED.ent_id, RANKED["warm intros"]))).fillna("")
else:
    G["intros"] = ""
if len(INTRO):
    t3 = (INTRO.sort_values(["ent_id", "total"], ascending=[True, False]).groupby("ent_id").head(3)
               .groupby("ent_id").apply(lambda d: "; ".join(f"{n} ({t / 1e3:,.0f}K)" for n, t in zip(d.tm_client, d.total))))
    G["intros"] = G.intros.where(G.intros != "", G.ent_id.map(t3).fillna(""))
PROS = G[G.tier.isin(["A", "B"])].head(N_PROSPECTS)
if len(PROS) < N_PROSPECTS:
    PROS = pd.concat([PROS, G[G.tier == "C"].head(N_PROSPECTS - len(PROS))])
BRANDS = CP[(CP.relationship == "registered TPPP brand") & ~CP.tppp_ext].sort_values("amt_in", ascending=False).head(N_BRANDS)
CARD = CP[CP.card_settlement]
BANKS = (CARD.groupby(CARD.fi_dom.fillna("(unknown)"))
             .agg(descriptors=("ent_id", "size"), clients_paid=("ncust_out", "sum"), amt_out=("amt_out", "sum"))
             .reset_index().rename(columns={"fi_dom": "bank"}).sort_values("clients_paid", ascending=False).head(N_BANKS))

# ---- look-alikes ----
pool_c = CS[~CS.is_tppp & ~CS.tppp_family & ~CS.excluded & (~CS.sector_unlikely | CS.declared)]
LOOK = pool_c.sort_values("score", ascending=False).head(N_LOOKALIKES)
DECL = pool_c[pool_c.triage == "declared only"].sort_values("amt_in", ascending=False).head(N_DECLARED)

# ---- reliability ----
def best(rec, group):
    if rec is None:
        return None
    r = rec[(rec.examples == group) & rec.method.isin(["gbm", "logit", "knn", "blend"])]
    return r.sort_values("median % ranked above").iloc[0] if len(r) else None
BEST_C, BEST_P = best(REC_C, "all examples"), best(REC_P, "known processors")
N_SCREENED_C = int(len(CS))
N_SCREENED_P = int(len(CP))

counts = pd.DataFrame([
    {"item": "registered TPPPs shown (business elsewhere)", "n": 0 if REG is None else len(REG)},
    {"item": "prospects ranked (fit gate passed)", "n": len(G)},
    {"item": "prospects tier A / B / C", "n": " / ".join(str(int((G.tier == t).sum())) for t in "ABC")},
    {"item": "prospects shown", "n": len(PROS)},
    {"item": "registered TPPP brands shown", "n": len(BRANDS)},
    {"item": "card-settlement banks shown", "n": len(BANKS)},
    {"item": "look-alikes shown / confirmed by data", "n": f"{len(LOOK)} / {int(LOOK.data_confirmed.sum())}"},
    {"item": "declared-only (KYC) shown", "n": len(DECL)},
])
show("§2 Report contents", ("Counts", counts))

In [ ]:
# §3  HTML helpers: formatting, tables, charts
E = html.escape

def usd(v):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "–"
    a = abs(v)
    if a >= 1e9:
        return f"${v / 1e9:,.1f}B"
    if a >= 1e6:
        return f"${v / 1e6:,.1f}M"
    if a >= 1e3:
        return f"${v / 1e3:,.0f}K"
    return f"${v:,.0f}"

def num(v, d=0):
    return "–" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{v:,.{d}f}"

def pct(v, d=0):
    return "–" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{100 * v:.{d}f}%"

class Cell:
    # a table cell: shown html, sort value, alignment
    def __init__(self, show, sort=None, num=False, raw=False):
        self.show, self.sort, self.num, self.raw = show, sort, num, raw

def C(v, kind="text", d=0):
    if kind == "usd":
        return Cell(usd(v), v if v is not None and np.isfinite(v) else -1e18, True)
    if kind == "num":
        return Cell(num(v, d), v if v is not None and np.isfinite(v) else -1e18, True)
    if kind == "pct":
        return Cell(pct(v, d), v if v is not None and np.isfinite(v) else -1e18, True)
    if kind == "html":
        return Cell(v, None, False, True)
    s = "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v)
    return Cell(s, s.lower())

def badge(text, kind):
    return f"<span class='b b-{kind}'>{E(text)}</span>"

def table(tid, headers, rows, search=True, chips=None, note=None, height=None):
    th = "".join(f"<th data-i='{i}' class='{'n' if any(isinstance(r[i], Cell) and r[i].num for r in rows[:1]) else ''}'>"
                 f"{E(h)}<span class='ar'></span></th>" for i, h in enumerate(headers))
    body = []
    for r in rows:
        tds = []
        for c in r:
            c = c if isinstance(c, Cell) else C(c)
            sv = "" if c.sort is None else f" data-v='{E(str(c.sort))}'"
            cls = "n" if c.num else ("wrap" if not c.raw and len(c.show) > 50 else "")
            tds.append(f"<td class='{cls}'{sv}>{c.show if c.raw else E(c.show)}</td>")
        body.append("<tr>" + "".join(tds) + "</tr>")
    tools = ""
    if search or chips:
        tools = "<div class='tools'>"
        if search:
            tools += f"<input type='search' class='q' data-t='{tid}' placeholder='Search this table…' aria-label='Search'>"
        for label, col, vals in (chips or []):
            tools += f"<span class='cg' data-t='{tid}' data-c='{col}'><span class='cl'>{E(label)}</span>"
            tools += "".join(f"<button class='chip on' data-val='{E(v)}'>{E(v)}</button>" for v in vals) + "</span>"
        tools += f"<span class='cnt' id='{tid}-n'>{len(rows)} rows</span></div>"
    st = f" style='max-height:{height}px'" if height else ""
    return (tools + f"<div class='tw'{st}><table id='{tid}' class='dt'><thead><tr>{th}</tr></thead>"
            f"<tbody>{''.join(body)}</tbody></table></div>" + (f"<p class='note'>{note}</p>" if note else ""))

def svg_share(reg, title_id):
    W, left, right, top, rowh = 820, 250, 24, 44, 34
    H = top + rowh * len(reg) + 30
    x0, x1 = left, W - right
    p = [f"<svg viewBox='0 0 {W} {H}' class='chart' role='img' aria-labelledby='{title_id}'>",
         f"<rect x='{left}' y='10' width='12' height='12' rx='3' class='f1'/><text class='lab' x='{left + 18}' y='20'>paid into its PNC accounts</text>",
         f"<rect x='{left + 200}' y='10' width='12' height='12' rx='3' class='f2'/><text class='lab' x='{left + 218}' y='20'>paid into its accounts at other banks</text>"]
    for t in [0, 25, 50, 75, 100]:
        x = x0 + (x1 - x0) * t / 100
        p.append(f"<line class='{'base' if t == 0 else 'grid'}' x1='{x:.1f}' x2='{x:.1f}' y1='{top - 6}' y2='{H - 26}'/>"
                 f"<text class='tick' x='{x:.1f}' y='{H - 10}' text-anchor='middle'>{t}%</text>")
    for i, r in reg.iterrows():
        y = top + i * rowh
        tot = r.pnc + r.out
        if not tot:
            continue
        wp = (x1 - x0) * r.pnc / tot
        tip = (f"{r.tppp}: {usd(r.pnc)}/mo into PNC accounts, {usd(r.out)}/mo elsewhere "
               f"({pct(r.share)} elsewhere, {MONTHS_ST[0]}…{MONTHS_ST[-1]})")
        name = re.sub(r"\b(INC|LLC|CORP|CO|LTD)\b\.?", "", str(r.tppp)).strip().title()
        g = [f"<g class='hit' tabindex='0' data-tip='{E(tip)}'><rect x='0' y='{y}' width='{W}' height='{rowh}' fill='transparent'/>",
             f"<text class='lab' x='{left - 10}' y='{y + 19}' text-anchor='end'>{E(name[:34])}</text>"]
        if wp > 0.5:
            g.append(f"<path class='f1' d='M{x0:.1f},{y + 5} H{x0 + wp:.1f} V{y + 25} H{x0:.1f} Z'/>")
        if (x1 - x0) - wp > 3:
            xs = x0 + wp + (2 if wp > 0.5 else 0)
            g.append(f"<path class='f2' d='M{xs:.1f},{y + 5} H{x1 - 4:.1f} Q{x1:.1f},{y + 5} {x1:.1f},{y + 9} V{y + 21} "
                     f"Q{x1:.1f},{y + 25} {x1 - 4:.1f},{y + 25} H{xs:.1f} Z'/>")
        if (x1 - x0) - wp > 150:
            g.append(f"<text class='in dk' x='{x1 - 8:.1f}' y='{y + 19}' text-anchor='end'>{E(usd(r.out))}/mo · {pct(r.share)}</text>")
        if wp > 80:
            g.append(f"<text class='in' x='{x0 + 8:.1f}' y='{y + 19}'>{E(usd(r.pnc))}</text>")
        g.append("</g>")
        p.append("".join(g))
    p.append("</svg>")
    return "".join(p)

def svg_recall(items, title_id):
    # items: list of (label, found_1, found_5, n)
    W, left, right, top, grp = 820, 250, 60, 40, 50
    H = top + grp * len(items) + 28
    x0, x1 = left, W - right
    p = [f"<svg viewBox='0 0 {W} {H}' class='chart' role='img' aria-labelledby='{title_id}'>",
         f"<rect x='{left}' y='10' width='12' height='12' rx='3' class='f1'/><text class='lab' x='{left + 18}' y='20'>found in the top 1%</text>",
         f"<rect x='{left + 160}' y='10' width='12' height='12' rx='3' class='f2'/><text class='lab' x='{left + 178}' y='20'>found in the top 5%</text>"]
    for t in [0, 25, 50, 75, 100]:
        x = x0 + (x1 - x0) * t / 100
        p.append(f"<line class='{'base' if t == 0 else 'grid'}' x1='{x:.1f}' x2='{x:.1f}' y1='{top - 6}' y2='{H - 26}'/>"
                 f"<text class='tick' x='{x:.1f}' y='{H - 10}' text-anchor='middle'>{t}%</text>")
    for i, (lab, f1, f5, n) in enumerate(items):
        y = top + i * grp
        tip = f"{lab}: {num(f1, 1 if f1 < 10 and f1 % 1 else 0)} of {n} in the top 1%, {num(f5, 1 if f5 < 10 and f5 % 1 else 0)} in the top 5%"
        g = [f"<g class='hit' tabindex='0' data-tip='{E(tip)}'><rect x='0' y='{y}' width='{W}' height='{grp}' fill='transparent'/>",
             f"<text class='lab' x='{left - 10}' y='{y + 24}' text-anchor='end'>{E(lab)}</text>"]
        for j, (v, cls) in enumerate([(f1, "f1"), (f5, "f2")]):
            yy = y + 5 + j * 19
            w = max((x1 - x0) * v / n, 1.5)
            r_ = min(4, w)
            g.append(f"<path class='{cls}' d='M{x0:.1f},{yy} H{x0 + w - r_:.1f} Q{x0 + w:.1f},{yy} {x0 + w:.1f},{yy + r_} "
                     f"V{yy + 16 - r_} Q{x0 + w:.1f},{yy + 16} {x0 + w - r_:.1f},{yy + 16} H{x0:.1f} Z'/>")
            g.append(f"<text class='val' x='{x0 + w + 6:.1f}' y='{yy + 12}'>{pct(v / n)}</text>")
        g.append("</g>")
        p.append("".join(g))
    p.append("</svg>")
    return "".join(p)

def svg_scatter(d, title_id):
    # prospect map: x = TM clients, y = money with PNC clients per month (log scales), colour = tier
    W, H, l, r, t, b = 820, 460, 70, 20, 40, 50
    d = d[(d.tm_clients > 0) & (d.amt_stable > 0)].copy()
    if not len(d):
        return "<p class='note'>No prospects with both measures.</p>"
    keep = d[d.tier.isin(["A", "B"])]
    rest = d[~d.tier.isin(["A", "B"])]
    if len(rest) > SCATTER_MAX - len(keep):
        rest = rest.sample(max(SCATTER_MAX - len(keep), 0), random_state=1)
    d = pd.concat([rest, keep])            # A and B drawn last (on top)
    lx, ly = np.log10(d.tm_clients), np.log10(d.amt_stable)
    xa, xb = math.floor(lx.min()), math.ceil(lx.max() + 1e-9)
    ya, yb = math.floor(ly.min()), math.ceil(ly.max() + 1e-9)
    xb, yb = max(xb, xa + 1), max(yb, ya + 1)
    sx = lambda v: l + (W - l - r) * (v - xa) / (xb - xa)
    sy = lambda v: H - b - (H - t - b) * (v - ya) / (yb - ya)
    p = [f"<svg viewBox='0 0 {W} {H}' class='chart' role='img' aria-labelledby='{title_id}'>"]
    for k in range(xa, xb + 1):
        p.append(f"<line class='grid' x1='{sx(k):.1f}' x2='{sx(k):.1f}' y1='{t}' y2='{H - b}'/>"
                 f"<text class='tick' x='{sx(k):.1f}' y='{H - b + 18}' text-anchor='middle'>{num(10 ** k)}</text>")
    for k in range(ya, yb + 1):
        p.append(f"<line class='grid' x1='{l}' x2='{W - r}' y1='{sy(k):.1f}' y2='{sy(k):.1f}'/>"
                 f"<text class='tick' x='{l - 8}' y='{sy(k) + 4:.1f}' text-anchor='end'>{usd(10 ** k)}</text>")
    p.append(f"<text class='lab' x='{(W + l) / 2:.0f}' y='{H - 10}' text-anchor='middle'>TM clients that pay it or are paid by it (log scale)</text>"
             f"<text class='lab' transform='translate(16,{(H - b + t) / 2:.0f}) rotate(-90)' text-anchor='middle'>money with PNC clients per month (log)</text>")
    lg = [("A", "dA", "tier A"), ("B", "dB", "tier B"), ("C", "dC", "tier C")]
    for i, (_, cls, lab) in enumerate(lg):
        p.append(f"<circle class='{cls}' cx='{l + 10 + i * 90}' cy='18' r='5'/><text class='lab' x='{l + 20 + i * 90}' y='22'>{lab}</text>")
    for _, row in d.iterrows():
        cls = {"A": "dA", "B": "dB"}.get(row.tier, "dC")
        rad = 5 if row.tier in ("A", "B") else 3
        tip = (f"{row.ent_name} · tier {row.tier} · {row.relationship} · {usd(row.amt_stable)}/mo · "
               f"{num(row.tm_clients)} TM clients · bank today {row.fi_dom}")
        p.append(f"<circle class='{cls} pt' cx='{sx(np.log10(row.tm_clients)):.1f}' cy='{sy(np.log10(row.amt_stable)):.1f}' "
                 f"r='{rad}' tabindex='0' data-tip='{E(tip)}'/>")
    placed = []                              # label the strongest A/B points, skipping any that would collide
    for _, row in keep.sort_values("consensus", ascending=False).iterrows():
        if len(placed) >= 6:
            break
        x, y = sx(np.log10(row.tm_clients)), sy(np.log10(row.amt_stable))
        txt = str(row.ent_name)[:24]
        wdt = 6.6 * len(txt)
        anchor = "end" if x > W * 0.65 else "start"
        x0_ = x - 8 - wdt if anchor == "end" else x + 8
        box = (x0_, y - 19, x0_ + wdt, y - 5)
        if any(not (box[2] < b2[0] or box[0] > b2[2] or box[3] < b2[1] or box[1] > b2[3]) for b2 in placed):
            continue
        placed.append(box)
        p.append(f"<line class='base' x1='{x:.1f}' y1='{y:.1f}' x2='{(x - 6 if anchor == 'end' else x + 6):.1f}' y2='{y - 9:.1f}'/>"
                 f"<text class='pl' x='{(x - 8 if anchor == 'end' else x + 8):.1f}' y='{y - 8:.1f}' text-anchor='{anchor}'>{E(txt)}</text>")
    p.append("</svg>")
    return "".join(p)
print("helpers ready")

In [ ]:
# §4  Build and write the report
CSS = '''
:root{color-scheme:light;--page:#f6f5f1;--surf:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--mut:#7d7c76;--grid:#e1e0d9;
--base:#c3c2b7;--ring:rgba(11,11,11,.10);--s1:#2a78d6;--s2:#eb6834;--s3:#1baf7a;--dC:#c3c2b7;--wash:#eef4fc;
--warm:#fdf3ea;--th:#efeee9;--acc:#104281;--good:#006300}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){color-scheme:dark;--page:#0d0d0d;--surf:#1a1a19;
--ink:#f0efec;--ink2:#c3c2b7;--mut:#8f8e88;--grid:#2c2c2a;--base:#383835;--ring:rgba(255,255,255,.10);--s1:#3987e5;
--s2:#d95926;--s3:#199e70;--dC:#4a4a46;--wash:#16222f;--warm:#2a1d14;--th:#242422;--acc:#86b6ef;--good:#0ca30c}}
:root[data-theme="dark"]{color-scheme:dark;--page:#0d0d0d;--surf:#1a1a19;--ink:#f0efec;--ink2:#c3c2b7;--mut:#8f8e88;
--grid:#2c2c2a;--base:#383835;--ring:rgba(255,255,255,.10);--s1:#3987e5;--s2:#d95926;--s3:#199e70;--dC:#4a4a46;
--wash:#16222f;--warm:#2a1d14;--th:#242422;--acc:#86b6ef;--good:#0ca30c}
*{box-sizing:border-box}html{scroll-behavior:smooth}
body{margin:0;background:var(--page);color:var(--ink);font:15px/1.55 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
.bar{background:var(--acc);color:#fff;font-size:12.5px;padding:6px 16px;text-align:center;letter-spacing:.02em}
:root[data-theme="dark"] .bar{color:#0b0b0b}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]) .bar{color:#0b0b0b}}
header.hero{max-width:1200px;margin:0 auto;padding:36px 16px 10px}
.eyebrow{color:var(--mut);font-size:12.5px;letter-spacing:.06em;text-transform:uppercase}
h1{font-size:32px;line-height:1.15;margin:8px 0 8px;letter-spacing:-.01em}
.lede{color:var(--ink2);max-width:820px;margin:0}
nav.toc{position:sticky;top:0;z-index:5;background:color-mix(in srgb,var(--page) 92%,transparent);backdrop-filter:blur(6px);
border-bottom:1px solid var(--grid)}
nav.toc div{max-width:1200px;margin:0 auto;padding:8px 16px;display:flex;gap:6px;overflow-x:auto;white-space:nowrap}
nav.toc a{color:var(--ink2);text-decoration:none;font-size:13.5px;padding:4px 10px;border-radius:14px}
nav.toc a:hover{background:var(--th);color:var(--ink)}
main{max-width:1200px;margin:0 auto;padding:8px 16px 80px}
section{margin-top:44px;scroll-margin-top:52px}
h2{font-size:22px;margin:0 0 4px} h3{font-size:16px;margin:22px 0 6px}
.sub{color:var(--ink2);margin:0 0 12px;max-width:900px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;margin:22px 0 4px}
.kpi{background:var(--surf);border:1px solid var(--ring);border-radius:12px;padding:16px 18px}
.kpi .v{font-size:30px;font-weight:650;line-height:1.1;letter-spacing:-.01em}
.kpi .l{color:var(--ink2);font-size:13.5px;margin-top:6px}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:12px;margin-top:12px}
.card{background:var(--surf);border:1px solid var(--ring);border-radius:12px;padding:16px 18px;border-top:4px solid var(--s1)}
.card.c2{border-top-color:var(--s2)} .card.c3{border-top-color:var(--s3)}
.card .who{font-size:12px;text-transform:uppercase;letter-spacing:.05em;color:var(--mut)}
.card h3{margin:4px 0 6px} .card p{margin:0 0 8px;color:var(--ink2);font-size:14px}
.card a{color:var(--s1);font-weight:600;text-decoration:none;font-size:14px}
figure{margin:12px 0;background:var(--surf);border:1px solid var(--ring);border-radius:12px;padding:14px 16px 8px}
figcaption{font-weight:600} .cap{color:var(--ink2);font-size:13.5px;margin:2px 0 6px}
svg.chart{width:100%;height:auto;display:block;font:12px system-ui,-apple-system,"Segoe UI",sans-serif}
svg .grid{stroke:var(--grid);stroke-width:1} svg .base{stroke:var(--base);stroke-width:1}
svg .tick{fill:var(--mut)} svg .lab{fill:var(--ink2)} svg .val{fill:var(--ink);font-weight:600}
svg .in{fill:#fff;font-weight:600} svg .in.dk{fill:#0b0b0b} svg .pl{fill:var(--ink);font-weight:600;font-size:11.5px}
svg .f1{fill:var(--s1)} svg .f2{fill:var(--s2)}
svg .dA{fill:var(--s1);stroke:var(--surf);stroke-width:2} svg .dB{fill:var(--s2);stroke:var(--surf);stroke-width:2}
svg .dC{fill:var(--dC)} svg .pt:hover,svg .pt:focus{stroke:var(--ink);stroke-width:2;outline:none}
svg .hit:focus{outline:none} svg .hit:hover .lab,svg .hit:focus .lab{fill:var(--ink)}
.tools{display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin:10px 0 6px}
.q{font:inherit;font-size:13.5px;padding:6px 10px;border:1px solid var(--ring);border-radius:8px;background:var(--surf);
color:var(--ink);min-width:240px}
.cg{display:inline-flex;gap:4px;align-items:center;flex-wrap:wrap}.cl{font-size:12.5px;color:var(--mut);margin-right:2px}
.chip{font:inherit;font-size:12.5px;padding:3px 10px;border-radius:14px;border:1px solid var(--ring);background:transparent;
color:var(--mut);cursor:pointer}
.chip.on{background:var(--th);color:var(--ink);border-color:var(--base)}
.cnt{margin-left:auto;font-size:12.5px;color:var(--mut)}
.tw{overflow:auto;border:1px solid var(--ring);border-radius:10px;background:var(--surf);max-height:620px}
table.dt{border-collapse:separate;border-spacing:0;width:100%;font-size:13px}
.dt th{position:sticky;top:0;background:var(--th);color:var(--ink2);font-weight:600;text-align:left;padding:7px 10px;
white-space:nowrap;cursor:pointer;user-select:none;border-bottom:1px solid var(--grid);z-index:1}
.dt td{padding:6px 10px;border-bottom:1px solid var(--grid);vertical-align:top}
.dt tr:hover td{background:var(--wash)} .dt .n{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}
.dt td.wrap{min-width:240px;max-width:420px} .ar{font-size:10px;margin-left:4px;color:var(--mut)}
.b{display:inline-block;font-size:11.5px;font-weight:600;padding:1px 8px;border-radius:10px;white-space:nowrap}
.b-A{background:var(--s1);color:#fff} .b-B{background:var(--s2);color:#fff} .b-C{background:var(--th);color:var(--ink2)}
.b-new{background:var(--wash);color:var(--ink)} .b-ex{background:var(--warm);color:var(--ink)} .b-g{background:var(--th);color:var(--ink2)}
.b-ok{background:var(--wash);color:var(--good)}
.note{color:var(--mut);font-size:12.5px;margin:6px 0 0}
.box{border-left:3px solid var(--s2);background:var(--warm);padding:10px 14px;border-radius:0 8px 8px 0;margin:12px 0;font-size:14px}
.box.i{border-left-color:var(--s1);background:var(--wash)}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:14px} @media (max-width:860px){.grid2{grid-template-columns:1fr}}
dl.gl{display:grid;grid-template-columns:200px 1fr;gap:6px 16px;margin:8px 0} dl.gl dt{font-weight:600} dl.gl dd{margin:0;color:var(--ink2)}
@media (max-width:640px){dl.gl{grid-template-columns:1fr} h1{font-size:25px} .q{min-width:0;width:100%}}
details{margin:10px 0} summary{cursor:pointer;color:var(--s1);font-weight:600;font-size:14px}
#tip{position:fixed;pointer-events:none;background:var(--ink);color:var(--surf);font-size:12.5px;padding:6px 9px;border-radius:6px;
max-width:360px;opacity:0;transition:opacity .08s;z-index:20}
footer{max-width:1200px;margin:0 auto;padding:18px 16px 40px;color:var(--mut);font-size:12.5px;border-top:1px solid var(--grid)}
@media print{nav.toc,.tools{display:none}.tw{max-height:none;overflow:visible}}
'''

JS = '''
const tip=document.getElementById('tip');
function showTip(e,el){tip.textContent=el.dataset.tip;tip.style.opacity=1;const r=el.getBoundingClientRect();
 const x=e&&e.clientX?e.clientX:r.left+r.width/2,y=e&&e.clientY?e.clientY:r.top;
 tip.style.left=Math.max(8,Math.min(x+12,innerWidth-tip.offsetWidth-8))+'px';tip.style.top=Math.max(y-tip.offsetHeight-10,8)+'px'}
document.querySelectorAll('[data-tip]').forEach(el=>{el.addEventListener('mousemove',e=>showTip(e,el));
 el.addEventListener('mouseleave',()=>tip.style.opacity=0);el.addEventListener('focus',()=>showTip(null,el));
 el.addEventListener('blur',()=>tip.style.opacity=0)});
function refilter(id){const t=document.getElementById(id);if(!t)return;
 const q=(document.querySelector(`.q[data-t="${id}"]`)||{value:''}).value.toLowerCase();
 const groups=[...document.querySelectorAll(`.cg[data-t="${id}"]`)].map(g=>({c:+g.dataset.c,
   on:new Set([...g.querySelectorAll('.chip.on')].map(b=>b.dataset.val))}));
 let n=0;t.tBodies[0].querySelectorAll('tr').forEach(tr=>{const txt=tr.textContent.toLowerCase();
  let ok=!q||txt.includes(q);groups.forEach(g=>{const v=tr.cells[g.c].textContent.trim();if(!g.on.has(v))ok=false});
  tr.style.display=ok?'':'none';if(ok)n++});
 const c=document.getElementById(id+'-n');if(c)c.textContent=n+' rows'}
document.querySelectorAll('.q').forEach(i=>i.addEventListener('input',()=>refilter(i.dataset.t)));
document.querySelectorAll('.cg .chip').forEach(b=>b.addEventListener('click',()=>{b.classList.toggle('on');
 refilter(b.closest('.cg').dataset.t)}));
document.querySelectorAll('table.dt').forEach(t=>{t.querySelectorAll('th').forEach(th=>th.addEventListener('click',()=>{
 const i=+th.dataset.i,asc=th.dataset.d!=='asc';t.querySelectorAll('th').forEach(h=>{h.dataset.d='';h.querySelector('.ar').textContent=''});
 th.dataset.d=asc?'asc':'desc';th.querySelector('.ar').textContent=asc?'▲':'▼';
 const rows=[...t.tBodies[0].rows];rows.sort((a,b)=>{const x=a.cells[i],y=b.cells[i];
  const xv=x.dataset.v!==undefined?x.dataset.v:x.textContent.trim().toLowerCase(),yv=y.dataset.v!==undefined?y.dataset.v:y.textContent.trim().toLowerCase();
  const xn=parseFloat(xv),yn=parseFloat(yv);const r=(!isNaN(xn)&&!isNaN(yn))?xn-yn:String(xv).localeCompare(String(yv));
  return asc?r:-r});rows.forEach(r=>t.tBodies[0].appendChild(r))}))});
'''

REL_BADGE = {"new to bank": "new", "existing PNC client": "ex", "possible PNC client": "ex", "registered TPPP brand": "g"}
def rel_badge(r):
    return badge(r, REL_BADGE.get(r, "g"))

parts = []
A = parts.append
gen = dt.datetime.now().strftime("%d %B %Y")
A(f"<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>"
  f"<title>TPPP Opportunities Report</title><style>{CSS}</style></head><body>"
  f"<div class='bar'>Internal · restricted to the {E(AUDIENCE)} · contains client names · prospecting use pending the "
  f"permissible-use review</div>"
  f"<header class='hero'><div class='eyebrow'>PNC Treasury Management · Payment Knowledge Graph · {E(VERSION)} · "
  f"generated {gen}</div><h1>{E(TITLE)}</h1>"
  f"<p class='lede'>Where third-party payment and payroll processors (TPPPs) show up in our clients' payments, "
  f"window {E(WINDOW_TXT.replace('..', ' to '))}. Three opportunities: registered TPPPs whose PNC-client money lands "
  f"at other banks, outside processors we could win, and our own clients that behave like processors.</p></header>")

nav = [("start", "Start here"), ("registered", "Registered TPPPs"), ("prospects", "Prospects"),
       ("lookalikes", "Look-alikes"), ("cards", "Card settlement"), ("reliability", "How reliable"),
       ("method", "How it works")]
A("<nav class='toc'><div>" + "".join(f"<a href='#{a}'>{E(b)}</a>" for a, b in nav) + "</div></nav><main>")

# ---------------- KPIs and start cards ----------------
k = []
if REG is not None and len(REG):
    k.append((usd(REG.out.sum()) + " / mo", f"paid by PNC clients into {len(REG)} registered TPPPs' accounts at other "
              f"banks ({pct(REG.out.sum() / (REG.out.sum() + REG.pnc.sum()))} of what they pay them), "
              f"{MONTHS_ST[0]} to {MONTHS_ST[-1]}"))
nAB_new = int(((G.tier.isin(["A", "B"])) & (G.relationship == "new to bank")).sum())
k.append((f"{int((G.tier == 'A').sum())} + {int((G.tier == 'B').sum())}",
          f"tier A + B prospects among {len(G):,} processor-like outside companies; {nAB_new} of them new to bank"))
k.append((f"{int(LOOK.data_confirmed.sum())} of {len(LOOK)}",
          "top look-alikes (our clients behaving like TPPPs) confirmed by at least two independent data checks"))
if BEST_C is not None:
    c1 = [c for c in REC_C.columns if c.startswith("found in top 1%")][0]
    k.append((f"{int(BEST_C[c1])} of {int(BEST_C.n)}", "registered TPPPs found again in the top 1% of business clients "
              "when hidden from the method (random review would find about 1%)"))
A("<section id='start' style='margin-top:10px'><div class='kpis'>" +
  "".join(f"<div class='kpi'><div class='v'>{E(v)}</div><div class='l'>{E(l)}</div></div>" for v, l in k) + "</div>")
A("<div class='cards'>"
  "<div class='card'><div class='who'>Relationship managers</div><h3>Bring registered TPPPs' money home</h3>"
  "<p>PNC clients already pay these processors; most of the money lands at another bank. The relationship exists, "
  "the amount and the competing bank are known.</p><a href='#registered'>See the registered TPPPs →</a></div>"
  "<div class='card c2'><div class='who'>TM sales</div><h3>Win outside processors</h3>"
  f"<p>{nAB_new} tier A–B prospects bank elsewhere today; each comes with the PNC clients who already pay it or are "
  "paid by it — the warm introduction.</p><a href='#prospects'>See the prospects →</a></div>"
  "<div class='card c3'><div class='who'>TPPP programme · KYC</div><h3>Check clients that act like processors</h3>"
  f"<p>Clients that move money like registered TPPPs but are not registered, plus {len(DECL)} that declare a "
  "processor industry code without behaving like one.</p><a href='#lookalikes'>See the look-alikes →</a></div></div></section>")

# ---------------- registered ----------------
A("<section id='registered'><h2>Registered TPPPs: business at other banks</h2>"
  "<p class='sub'>For each registered TPPP, money PNC clients pay into its PNC accounts compared with accounts carrying "
  "its name at other banks. Monthly averages over the stable months.</p>")
if REG is not None and len(REG):
    A("<figure><figcaption id='f-reg'>Where PNC clients pay the largest registered TPPPs</figcaption>"
      f"<p class='cap'>Share of monthly money, {MONTHS_ST[0]} to {MONTHS_ST[-1]}. Hover a bar for amounts.</p>"
      + svg_share(REG, "f-reg") + "</figure>")
    rows = [[C(r.tppp), C(r.pnc, "usd"), C(r.out, "usd"), C(r.share, "pct"), C(r.share_first, "pct"),
             C(r.share_last, "pct"), C(r.share_late, "pct")] for r in REG.itertuples()]
    A(table("t-reg", ["Registered TPPP", "PNC accounts / mo", "Elsewhere / mo", "% elsewhere",
                      f"% elsewhere {MONTHS_ST[0]}", f"% elsewhere {MONTHS_ST[-1]}", "% elsewhere Jul–Aug"],
            rows, search=False,
            note="Name matching links an outside account to a TPPP; generic names can match wrongly. Jul–Aug is shown "
                 "only for comparison: total dollars in the source roughly doubled in those months on both sides."))
    up = REG[(REG.share_last < REG.share_first - 0.15)]
    if len(up):
        A("<div class='box i'><b>Moving toward PNC:</b> " + "; ".join(
            f"{E(r.tppp)} ({pct(r.share_first)} elsewhere in {MONTHS_ST[0]} → {pct(r.share_last)} in {MONTHS_ST[-1]})"
            for r in up.itertuples()) + ".</div>")
else:
    A("<p class='note'>The monthly file was not found; rerun §9b of the v5 notebook.</p>")
A("</section>")

# ---------------- prospects ----------------
A("<section id='prospects'><h2>Prospects: outside processors we could win</h2>"
  "<p class='sub'>Outside companies that move money like processors, ranked by how good a prospect they are: "
  "<b>wallet</b> (money with PNC clients), <b>reach</b> (TM clients), <b>steady</b> (recurring clients) and "
  "<b>growth</b>, averaged into a consensus. Tier A also needs strong evidence that it is a processor.</p>")
A("<figure><figcaption id='f-map'>Prospect map</figcaption><p class='cap'>Each dot is a processor-like outside "
  "company. Up and right: more money and more TM clients. Hover for details.</p>" + svg_scatter(G, "f-map") + "</figure>")
tiers = pd.crosstab(G.relationship, G.tier).reindex(columns=["A", "B", "C"], fill_value=0)
A("<div class='grid2'><div>" + table("t-tiers", ["Relationship to PNC", "Tier A", "Tier B", "Tier C"],
  [[C(i), C(int(r.A), "num"), C(int(r.B), "num"), C(int(r.C), "num")] for i, r in tiers.iterrows()], search=False)
  + "</div><div class='box i' style='margin:0'><b>Reading the relationship.</b> <i>New to bank</i>: no PNC client "
  "found by name. <i>Existing / possible PNC client</i>: same or similar name as a PNC organisation — the conversation "
  "is about bringing the rest of its business. Name matching misses clients known under another name; check before "
  "calling.</div></div>")
rows = []
for r in PROS.itertuples():
    rows.append([C(int(r.value_rank), "num"), C(badge(r.tier, r.tier), "html"), C(r.ent_name),
                 C(rel_badge(r.relationship), "html"), C(r.pnc_match_names if isinstance(r.pnc_match_names, str) else ""),
                 C("✓" if r.known_proc else ""), C(r.fi_dom), C(r.amt_stable, "usd"), C(r.tm_clients, "num"),
                 C(np.exp(r.growth) if pd.notna(r.growth) else None, "num", 2), C(r.value_reason), C(r.intros)])
A("<h3>Ranked prospects</h3>" + table(
    "t-pros", ["#", "Tier", "Company", "Relationship", "PNC name match", "Known", "Bank today", "Money / mo",
               "TM clients", "Growth ×", "Why", "Warm introductions"], rows,
    chips=[("Tier", 1, ["A", "B", "C"]), ("Relationship", 3, sorted(PROS.relationship.dropna().unique()))],
    note="Money / mo is with PNC clients, both directions, over the stable months. Growth × = April–June over "
         "January–March. Warm introductions are the TM clients moving the most money with the company."))
if len(BRANDS):
    rows = [[C(r.ent_name), C(r.pnc_match_names), C(r.fi_dom), C(r.amt_in, "usd"), C(r.ncust_in, "num")]
            for r in BRANDS.itertuples()]
    A("<details><summary>Accounts of registered TPPP brands at other banks (" + str(len(BRANDS)) +
      ") — for relationship managers</summary>" + table("t-brand", ["Outside account", "Brand of", "Bank today",
      "From PNC clients / mo", "PNC clients paying"], rows) + "</details>")
A("</section>")

# ---------------- look-alikes ----------------
A("<section id='lookalikes'><h2>Look-alikes: our clients behaving like TPPPs</h2>"
  "<p class='sub'>TM business clients that move money the way registered TPPPs do but are not registered. "
  "<b>Evidence</b> counts independent data checks passed (clients switching from a known processor to them, "
  "employers moving staff payments to them, and others) — only checks that registered TPPPs pass far more often than "
  "other clients are counted.</p>")
TRI = {"behaviour + declared": "A", "behaviour only": "B", "declared only": "C"}
rows = []
for i, r in enumerate(LOOK.itertuples(), 1):
    rows.append([C(i, "num"), C(r.customer_name), C(r.customer_pwr_id),
                 C(f"{r.naics_cd or ''} {str(r.naics_desc or '')[:34]}".strip()), C(r.triage or "–"),
                 C(badge(str(int(r.evidence_n)), "ok" if r.data_confirmed else "g"), "html"),
                 C(r.ncpty_in, "num"), C(r.ncpty_out, "num"), C(r.amt_in, "usd"), C(r.nearest_tppp),
                 C(r.evidence if isinstance(r.evidence, str) and r.evidence else r.reasons)])
A(table("t-look", ["#", "Client", "Power id", "NAICS", "Triage", "Evidence", "Payers / mo", "Payees / mo",
                   "Received / mo", "Most similar registered TPPP", "Why"], rows,
        chips=[("Triage", 4, sorted({x[4].show for x in rows}))],
        note="Triage: behaviour + declared = behaves like a TPPP and its industry code or name says processor; "
             "behaviour only = review; declared only = industry code says processor, behaviour does not."))
if len(DECL):
    rows = [[C(r.customer_name), C(r.customer_pwr_id), C(f"{r.naics_cd or ''} {str(r.naics_desc or '')[:34]}".strip()),
             C(r.amt_in, "usd"), C(r.ncpty_in, "num"), C(r.ncpty_out, "num")] for r in DECL.itertuples()]
    A("<h3>KYC check: declared processors that are not registered</h3><p class='sub'>Their industry code says "
      "payment processing but they are not registered as TPPPs. There may be good reasons (another client type, a "
      "funding entity); this is a classification question for the programme owners.</p>"
      + table("t-decl", ["Client", "Power id", "NAICS", "Received / mo", "Payers / mo", "Payees / mo"], rows))
A("</section>")

# ---------------- card settlement ----------------
A("<section id='cards'><h2>Card settlement from acquirers at other banks</h2><p class='sub'>Descriptors such as "
  "MERCHANT SERVICE or MERCH BNKCD are card deposits from merchant acquirers, named after the product, not the "
  "company. They are not TPPP prospects; they show which banks acquire card payments for PNC clients — useful to "
  "merchant services.</p>")
if len(BANKS):
    rows = [[C(r.bank), C(int(r.descriptors), "num"), C(r.clients_paid, "num"), C(r.amt_out, "usd")]
            for r in BANKS.itertuples()]
    A(table("t-bank", ["Bank holding the descriptor", "Descriptors", "PNC clients paid (summed)", "Paid to PNC clients / mo"],
            rows, search=False, note="A client receiving from several descriptors counts once per descriptor."))
else:
    A("<p class='note'>No card-settlement descriptors among the screened companies.</p>")
A("</section>")

# ---------------- reliability ----------------
A("<section id='reliability'><h2>How reliable are the results?</h2><p class='sub'>The method was tested the fair "
  "way: registered TPPPs and known processors were hidden, scored by a model built without them, and we counted how "
  "many came back near the top.</p>")
items = []
for rec, b_, lab, grp in [(REC_C, BEST_C, "Look-alikes: registered TPPPs", "all examples"),
                          (REC_P, BEST_P, "Prospects: known processors", "known processors")]:
    if b_ is not None:
        c1 = [c for c in rec.columns if c.startswith("found in top 1%")][0]
        c5 = [c for c in rec.columns if c.startswith("found in top 5%")][0]
        items.append((lab, float(b_[c1]), float(b_[c5]), int(b_.n)))
        items.append(("Random review (same list length)", 0.01 * int(b_.n), 0.05 * int(b_.n), int(b_.n)))
if items:
    A("<figure><figcaption id='f-rec'>Hidden examples found again near the top</figcaption><p class='cap'>Share of "
      "hidden examples that land in the top 1% and top 5% of the list.</p>" + svg_recall(items, "f-rec") + "</figure>")
if CHECKS is not None:
    rows = []
    for r in CHECKS.itertuples():
        lab = {"switch": "clients switched to it from a known processor", "payroll": "employers moved staff payments to it",
               "lead": "money in leads money out by 1–3 days", "shared": "its clients also use known processors",
               "data_confirmed": "confirmed overall (enough checks passed)"}.get(r.check, r.check)
        rows.append([C(r.list), C(lab), C(getattr(r, "_2") / 100, "pct", 1), C(getattr(r, "_3") / 100, "pct", 1),
                     C(r._4, "num", 1), C("✓" if str(r.used) == "True" else "")])
    A("<h3>Data checks: which ones are trusted</h3>" + table(
        "t-chk", ["List", "Check", "% of examples passing", "% of others passing", "Times as often", "Used"], rows,
        search=False, note="A check counts only when examples pass it at least three times as often as everyone else."))
A("</section>")

# ---------------- method ----------------
A("<section id='method'><h2>How it works</h2><div class='grid2'><div>"
  "<h3>In five steps</h3><ol>"
  "<li><b>Describe</b> every business client and every outside company by how its money moves: payers and payees, "
  "how regular they are, money in against money out, banks on the other side.</li>"
  "<li><b>Learn</b> from about 100 registered TPPPs and a list of well-known processors which of these patterns set "
  "processors apart.</li>"
  "<li><b>Score and test</b> every client and company; check the score on hidden examples.</li>"
  "<li><b>Confirm with data</b> that the score did not use: clients switching from a known processor, employers moving "
  "staff payments, money in leading money out.</li>"
  "<li><b>Rank prospects by value</b> — wallet, reach, steady relationships, growth — into tiers A, B, C.</li></ol></div>"
  "<div><h3>Words used</h3><dl class='gl'>"
  "<dt>TPPP</dt><dd>Third-party payment or payroll processor: moves money on behalf of other businesses.</dd>"
  "<dt>Look-alike</dt><dd>A PNC client that behaves like a registered TPPP but is not registered.</dd>"
  "<dt>Prospect</dt><dd>An outside company (a counterparty of our clients) that behaves like a processor.</dd>"
  "<dt>Tier A / B / C</dt><dd>Consensus of the four value measures: A ≥ 80th percentile with strong processor "
  "evidence, B ≥ 60th, C the rest.</dd>"
  "<dt>Warm introduction</dt><dd>A TM client already paying or paid by the prospect.</dd>"
  "<dt>Evidence</dt><dd>Number of independent data checks passed.</dd></dl></div></div>"
  "<div class='box'><b>Limits.</b> We see only the money that touches PNC clients. A high score is a reason to look, "
  "not a finding about a company. Names in payment feeds are messy, so relationships to PNC are matched by name. "
  "Amounts use January–June 2026: July and August carry a source-data jump under review.</div></section></main>")
A(f"<footer>Source: tppp_prospects_v5 outputs in {E(DB)} ({E(VERSION)}, window {E(WINDOW_TXT)}); generated "
  f"{gen} by tppp_report_v5.ipynb. Internal · restricted.</footer><div id='tip' role='tooltip'></div>"
  f"<script>{JS}</script></body></html>")

doc = "".join(parts)
assert "<script src" not in doc and "https://" not in doc, "report must stay self-contained"
os.makedirs(os.path.dirname(REPORT_FILE) or ".", exist_ok=True)
with open(REPORT_FILE, "w", encoding="utf-8") as f:
    f.write(doc)
show("§4 Report written",
     ("File", pd.DataFrame([{"path": os.path.abspath(REPORT_FILE), "size KB": round(len(doc.encode()) / 1024)}])),
     ("Sections", pd.DataFrame({"section": [b for _, b in nav]})),
     ("", "Open it from the JupyterLab file browser: right-click → Open in New Browser Tab, or download it."))